In [12]:
# Mount Drive, unzip data, fix paths, and test checkpoint — all in one cell
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

os.makedirs('/content/rtfm', exist_ok=True)

# Unzip feature data if not already present
if not os.path.exists('/content/rtfm/data'):
    !unzip -q /content/drive/MyDrive/rtfm_colab.zip -d /content/rtfm

# Copy Python source files from Drive if missing (model.py, dataset.py, option.py, etc.)
src_dir = '/content/drive/MyDrive/rtfm_src'
required_src = ['model.py', 'dataset.py', 'option.py', 'utils.py', 'config.py']
for fname in required_src:
    dst = f'/content/rtfm/{fname}'
    src = f'{src_dir}/{fname}'
    if not os.path.exists(dst):
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f'Copied {fname} from Drive.')
        else:
            print(f'WARNING: {fname} not found in Drive at {src} — import will fail.')

# Fix paths in list files
for list_file in ['shanghai-i3d-train-10crop.list', 'shanghai-i3d-test-10crop.list']:
    path = f'/content/rtfm/list/{list_file}'
    if not os.path.exists(path):
        print(f'WARNING: list file {list_file} not found, skipping path fix.')
        continue
    with open(path, 'r') as f:
        lines = f.readlines()
    new_lines = []
    for line in lines:
        fname = line.strip().split('/')[-1]
        if 'Train' in line:
            new_lines.append(f'/content/rtfm/data/SH_Train_ten_crop_i3d/{fname}\n')
        else:
            new_lines.append(f'/content/rtfm/data/SH_Test_ten_crop_i3d/{fname}\n')
    with open(path, 'w') as f:
        f.writelines(new_lines)

# Copy checkpoint from Drive if not already present
os.makedirs('/content/rtfm/ckpt', exist_ok=True)
ckpt_src = '/content/drive/MyDrive/rtfm_checkpoints/rtfm_best.pkl'
ckpt_dst = '/content/rtfm/ckpt/rtfm_best.pkl'
if not os.path.exists(ckpt_dst) and os.path.exists(ckpt_src):
    shutil.copy2(ckpt_src, ckpt_dst)

print('Setup done.')
print('Files in /content/rtfm:', [f for f in os.listdir('/content/rtfm') if f.endswith('.py')])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done.
Files in /content/rtfm: ['train_rtfm.py', 'option.py', 'utils.py', 'test_10crop.py', 'config.py', 'train.py', 'model.py', 'dataset.py']


In [16]:
# Test checkpoint AUC
import os, sys, importlib, importlib.util

rtfm_dir = '/content/rtfm'
assert os.path.isdir(rtfm_dir), f'{rtfm_dir} does not exist — run Cell 0 first.'

os.chdir(rtfm_dir)

# Force sys.path to include rtfm_dir regardless of prior state
for p in [rtfm_dir, '']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load modules directly by file path — avoids all sys.path caching issues on Colab
def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

utils   = _load('utils',   f'{rtfm_dir}/utils.py')
model   = _load('model',   f'{rtfm_dir}/model.py')
option  = _load('option',  f'{rtfm_dir}/option.py')
dataset = _load('dataset', f'{rtfm_dir}/dataset.py')
print('Imports OK')

import torch, numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc

args = option.parser.parse_args([])
device = torch.device('cuda')

net = model.Model(args.feature_size, args.batch_size)
ckpt = torch.load('ckpt/rtfm_best.pkl', map_location='cuda')
net.load_state_dict(ckpt)
net = net.to(device)
net.eval()

test_loader = DataLoader(dataset.Dataset(args, test_mode=True),
    batch_size=1, shuffle=False, num_workers=0)

pred = torch.zeros(0, device=device)
with torch.no_grad():
    for i, inp in enumerate(test_loader):
        inp = inp.to(device)
        inp = inp.permute(0, 2, 1, 3)
        _, _, _, _, _, _, logits, _, _, _ = net(inputs=inp)
        logits = torch.squeeze(logits, 1)
        logits = torch.mean(logits, 0)
        pred = torch.cat((pred, logits))

gt = np.load('list/gt-sh.npy')
pred_np = np.repeat(pred.cpu().numpy(), 16)

fpr, tpr, _ = roc_curve(gt, pred_np)
rec_auc = auc(fpr, tpr)
print(f'Checkpoint AUC on Colab (CUDA): {rec_auc:.4f}')

Imports OK
Checkpoint AUC on Colab (CUDA): 0.9637
